# Dataset 5 — Category Inflows Cleaning

## Objective
Clean and validate mutual fund category inflow data for further analysis.

## Source Dataset
`data/raw/5_category_inflows.csv`

## Output Dataset
`data/processed/cleaned_category_inflows.csv`

## Cleaning Workflow
1. Load and inspect the dataset.
2. Check column names and data types.
3. Identify missing values and duplicate records.
4. Validate dates and numeric values.
5. Handle data-quality issues appropriately.
6. Export and verify the cleaned dataset.

In [4]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd().parent

category_file = (
    project_path
    / "data"
    / "raw"
    / "05_category_inflows.csv"
)

category_df = pd.read_csv(category_file)

print("Dataset loaded successfully!")
print("Shape:", category_df.shape)

display(category_df.head())

Dataset loaded successfully!
Shape: (144, 3)


,month,category,net_inflow_crore
0,2024-04,Large Cap,2413.0
1,2024-04,Mid Cap,3897.0
2,2024-04,Small Cap,3533.0
3,2024-04,Flexi Cap,4947.0
4,2024-04,Large & Mid Cap,4214.0


In [5]:
# Inspect dataset structure
print("Dataset shape:", category_df.shape)

print("\nColumn names:")
print(category_df.columns.tolist())

print("\nData types:")
print(category_df.dtypes)

print("\nMissing values:")
print(category_df.isnull().sum())

print("\nDuplicate rows:", category_df.duplicated().sum())

print(
    "Duplicate month-category combinations:",
    category_df.duplicated(
        subset=["month", "category"]
    ).sum()
)

print("\nUnique categories:")
print(category_df["category"].unique())

print("\nReporting period:")
print("Earliest month:", category_df["month"].min())
print("Latest month:", category_df["month"].max())

print("\nNumeric summary:")
print(category_df["net_inflow_crore"].describe())

Dataset shape: (144, 3)

Column names:
['month', 'category', 'net_inflow_crore']

Data types:
month                   str
category                str
net_inflow_crore    float64
dtype: object

Missing values:
month               0
category            0
net_inflow_crore    0
dtype: int64

Duplicate rows: 0
Duplicate month-category combinations: 0

Unique categories:
<StringArray>
[        'Large Cap',           'Mid Cap',         'Small Cap',
         'Flexi Cap',   'Large & Mid Cap',              'ELSS',
      'Value/Contra', 'Sectoral/Thematic',            'Liquid',
    'Short Duration',              'Gilt',            'Hybrid']
Length: 12, dtype: str

Reporting period:
Earliest month: 2024-04
Latest month: 2025-03

Numeric summary:
count      144.000000
mean      6473.881944
std       9718.335947
min        437.000000
25%       1801.250000
50%       4148.000000
75%       5304.000000
max      41952.000000
Name: net_inflow_crore, dtype: float64


In [6]:
# Convert month strings temporarily for validation
category_check = category_df.copy()

category_check["month"] = pd.to_datetime(
    category_check["month"],
    format="%Y-%m",
    errors="coerce"
)

print("Invalid months:", category_check["month"].isna().sum())
print("Earliest month:", category_check["month"].min())
print("Latest month:", category_check["month"].max())

print("\nUnique categories:")
print(sorted(category_df["category"].unique()))

print("\nInflows less than or equal to zero:")
print((category_df["net_inflow_crore"] <= 0).sum())

print("\nTop 10 highest net inflows:")
display(
    category_df.nlargest(10, "net_inflow_crore")[
        ["month", "category", "net_inflow_crore"]
    ]
)

print("\nRecords per month:")
print(category_df.groupby("month").size().value_counts().sort_index())

Invalid months: 0
Earliest month: 2024-04-01 00:00:00
Latest month: 2025-03-01 00:00:00

Unique categories:
['ELSS', 'Flexi Cap', 'Gilt', 'Hybrid', 'Large & Mid Cap', 'Large Cap', 'Liquid', 'Mid Cap', 'Sectoral/Thematic', 'Short Duration', 'Small Cap', 'Value/Contra']

Inflows less than or equal to zero:
0

Top 10 highest net inflows:


,month,category,net_inflow_crore
56,2024-08,Liquid,41952.0
20,2024-05,Liquid,41872.0
92,2024-11,Liquid,40506.0
32,2024-06,Liquid,40486.0
80,2024-10,Liquid,39091.0
140,2025-03,Liquid,38681.0
8,2024-04,Liquid,37537.0
68,2024-09,Liquid,35308.0
104,2024-12,Liquid,34933.0
44,2024-07,Liquid,34643.0



Records per month:
12    12
Name: count, dtype: int64


In [7]:
# Create a copy of the original dataset
category_clean = category_df.copy()

# Convert month to datetime
category_clean["month"] = pd.to_datetime(
    category_clean["month"],
    format="%Y-%m",
    errors="coerce"
)

# Remove leading/trailing whitespace from category names
category_clean["category"] = (
    category_clean["category"].str.strip()
)

# Sort by month and category
category_clean = category_clean.sort_values(
    by=["month", "category"]
).reset_index(drop=True)

# Validate the cleaned dataset
print("Cleaned dataset shape:", category_clean.shape)
print("\nMissing values:")
print(category_clean.isnull().sum())

print("\nDuplicate rows:", category_clean.duplicated().sum())

print(
    "Duplicate month-category combinations:",
    category_clean.duplicated(
        subset=["month", "category"]
    ).sum()
)

print("Invalid months:", category_clean["month"].isnull().sum())

print("\nDate range:")
print("Earliest:", category_clean["month"].min())
print("Latest:", category_clean["month"].max())

print("\nCleaned preview:")
display(category_clean.head())

Cleaned dataset shape: (144, 3)

Missing values:
month               0
category            0
net_inflow_crore    0
dtype: int64

Duplicate rows: 0
Duplicate month-category combinations: 0
Invalid months: 0

Date range:
Earliest: 2024-04-01 00:00:00
Latest: 2025-03-01 00:00:00

Cleaned preview:


,month,category,net_inflow_crore
0,2024-04-01,ELSS,466.0
1,2024-04-01,Flexi Cap,4947.0
2,2024-04-01,Gilt,784.0
3,2024-04-01,Hybrid,2955.0
4,2024-04-01,Large & Mid Cap,4214.0


In [9]:
# Export the cleaned dataset
output_file = (
    project_path
    / "data"
    / "processed"
    / "cleaned_category_inflows.csv"
)

category_clean.to_csv(output_file, index=False)

print("Dataset exported successfully!")
print("Saved at:", output_file)
print("Final shape:", category_clean.shape)

Dataset exported successfully!
Saved at: d:\Mutual-Fund-Analytics\data\processed\cleaned_category_inflows.csv
Final shape: (144, 3)


In [10]:
# Reload the exported CSV to verify it
category_check = pd.read_csv(output_file)

print("Exported file shape:", category_check.shape)
print("\nMissing values:")
print(category_check.isnull().sum())

print("\nDuplicate rows:", category_check.duplicated().sum())

print("\nFirst 5 rows:")
display(category_check.head())

Exported file shape: (144, 3)

Missing values:
month               0
category            0
net_inflow_crore    0
dtype: int64

Duplicate rows: 0

First 5 rows:


,month,category,net_inflow_crore
0,2024-04-01,ELSS,466.0
1,2024-04-01,Flexi Cap,4947.0
2,2024-04-01,Gilt,784.0
3,2024-04-01,Hybrid,2955.0
4,2024-04-01,Large & Mid Cap,4214.0


## 📋 Data Cleaning Summary — Category Inflows

### Dataset Information
- **Source File:** `05_category_inflows.csv`
- **Output File:** `cleaned_category_inflows.csv`
- **Original Dataset:** 144 rows × 3 columns
- **Cleaned Dataset:** 144 rows × 3 columns
- **Reporting Period:** April 2024 – March 2025
- **Unique Categories:** 12

### 🧹 Cleaning Operations
1. Created a copy of the original dataset.
2. Converted the `month` column to datetime format.
3. Removed leading and trailing whitespace from category names.
4. Sorted records by month and category.
5. Checked for missing values and invalid dates.
6. Checked for duplicate rows and duplicate month-category combinations.
7. Exported the cleaned dataset to CSV.

### ✅ Validation Results
- Missing values: 0
- Duplicate rows: 0
- Duplicate month-category combinations: 0
- Invalid months: 0
- Final dataset shape: 144 rows × 3 columns
- Export verification: Completed

### 📁 Output Location
`data/processed/cleaned_category_inflows.csv`

### 🎯 Conclusion
The Category Inflows dataset was cleaned, validated, and exported successfully. It is ready for further analysis and integration into the Mutual Fund Analytics project.